# 04 — Desbalanceamento e limiar

## Objetivo

O desbalanceamento deve ser tratado por pesos, SMOTENC ou por uma decisão de
limiar?

Comparamos somente as três estratégias já validadas. Nenhum novo resampler é
adicionado.

In [ ]:
from pathlib import Path
import sys

ponto_atual = Path.cwd().resolve()
RAIZ = next(
    caminho for caminho in (ponto_atual, *ponto_atual.parents)
    if (caminho / "data" / "raw" / "UCI_Credit_Card.csv").exists()
)
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))


import json
import time
import pandas as pd
from imblearn.over_sampling import SMOTENC
from sklearn.utils.class_weight import compute_sample_weight

from src.auxiliares import (
    COLUNAS_NOMINAIS,
    PARAMETROS_REFERENCIA,
    avaliar_probabilidades,
    carregar_base_preparada,
    criar_modelo_gradiente,
    metricas_limiares,
    separar_dados,
)
from src.visual_utils import grafico_metricas_por_limiar

dados = carregar_base_preparada(RAIZ)
X_treino, X_validacao, X_teste, y_treino, y_validacao, y_teste = separar_dados(dados)
caminho_parametros = RAIZ / "models" / "parametros_gradient_boosting.json"
parametros = json.loads(caminho_parametros.read_text()) if caminho_parametros.exists() else PARAMETROS_REFERENCIA.copy()

## Como se comporta o modelo original?

## Pesos aumentam o alcance da classe positiva?

## O SMOTENC melhora o ranking?

## Qual estratégia preserva melhor a AP / PR-AUC?

In [ ]:
ap_original = resultados_balanceamento.query("modelo == 'Original'")["pr_auc"].iloc[0]
ap_alternativas = resultados_balanceamento.query("modelo != 'Original'")["pr_auc"]
ganho_maximo = ap_alternativas.max() - ap_original
estrategia_escolhida = "Original" if ganho_maximo <= 0.005 else "Reavaliar"

pd.Series({
    "estrategia_escolhida": estrategia_escolhida,
    "maior_ganho_alternativo_em_ap": ganho_maximo,
    "criterio_minimo": 0.005,
}).to_frame("resultado")

Pesos elevam Recall, mas também os falsos positivos. Um ganho de AP inferior a
0,005 não compensa essa mudança neste projeto. SMOTENC não precisa vencer para
ensinar: balancear classes não garante ranking melhor.

## O que muda quando alteramos o limiar?

In [ ]:
fig = grafico_metricas_por_limiar(resultados_limiares)
fig.show()

## Resultado

O modelo original permanece como solução final. O limiar 0,27 é uma escolha
ilustrativa, definida na prova técnica para aumentar Recall com perda de
Precision. Não há custo financeiro disponível para otimizá-lo.